# Application-Specific Tools: Practice Exercise

Build a production-grade book information tool for a LangChain agent using the **Open Library API** - a real, free API with millions of books.

**What you'll implement:**
- An API helper function that fetches data from the Open Library API
- A Pydantic model for book query validation
- A book search tool with proper error handling and logging

**Why Open Library?**
- Completely free, no API key required
- Real-world data (millions of books from libraries worldwide)
- Production-realistic API patterns (rate limits, network errors, data validation)

**Estimated time:** 20-25 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [ ]:
# Setup - run this cell first

import os
import logging
import time
from typing import Optional

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('book_tools')

print("Setup complete!")

## Part 1: Implement the API Helper Function

The Open Library API is a free, public API that provides access to book data from libraries worldwide.

**API Documentation:** https://openlibrary.org/developers/api

**Search Endpoint:** `https://openlibrary.org/search.json`

**Query Parameters:**
- `title`: The book title to search for
- `limit`: Number of results to return (use 1 for top result)
- `fields`: Comma-separated list of fields to return

**Example API Response:**
```json
{
  "numFound": 1234,
  "docs": [
    {
      "title": "The Great Gatsby",
      "author_name": ["F. Scott Fitzgerald"],
      "first_publish_year": 1925,
      "subject": ["Fiction", "American literature"],
      "publisher": ["Scribner"],
      "number_of_pages_median": 180
    }
  ]
}
```

**Your task:** Implement `fetch_book_data()` to:
1. Make a GET request to the Open Library Search API
2. Handle timeout errors (raise `TimeoutError`)
3. Handle other request errors (raise `ValueError`)
4. Check if any books were found (raise `ValueError` if not)
5. Extract and normalize the book data with graceful handling of missing fields

In [ ]:
def fetch_book_data(title: str) -> dict:
    """
    Fetch book data from the Open Library Search API.
    
    This is a REAL API call to openlibrary.org - no API key required!
    
    Args:
        title: The book title to search for
        
    Returns:
        dict: Book information with keys:
            - title (str): Book title
            - author (str): Author name(s), comma-separated
            - year (str): First publication year
            - subjects (list): Up to 5 subject categories
            - publisher (str): Publisher name
            - pages: Number of pages (or "Unknown")
        
    Raises:
        ValueError: If book is not found or API request fails
        TimeoutError: If the API request times out
    """
    logger.info(f"Searching Open Library for: {title}")
    
    # API endpoint and parameters
    url = "https://openlibrary.org/search.json"
    params = {
        "title": title,
        "limit": 1,  # Get only the top result
        "fields": "title,author_name,first_publish_year,subject,publisher,number_of_pages_median,key"
    }
    
    # TODO 1: Make the API request with a 10-second timeout
    # Use requests.get(url, params=params, timeout=10)
    # Then call response.raise_for_status() to check for HTTP errors
    
    # TODO 2: Wrap the request in a try/except block to handle:
    #   - requests.exceptions.Timeout -> raise TimeoutError with user-friendly message
    #   - requests.exceptions.RequestException -> raise ValueError with error details
    
    # TODO 3: Parse the JSON response with response.json()
    # Check if data["numFound"] == 0 and raise ValueError if no books found
    
    # TODO 4: Extract the first book from data["docs"][0]
    # Build and return a result dict with these keys:
    #   - "title": use book.get("title", "Unknown Title")
    #   - "author": join author_name list with ", " (default to ["Unknown Author"])
    #   - "year": convert first_publish_year to string (default to "Unknown")
    #   - "subjects": get first 5 subjects (default to empty list)
    #   - "publisher": get first publisher (default to "Unknown Publisher")
    #   - "pages": get number_of_pages_median (default to "Unknown")
    
    # Hint for author: ", ".join(book.get("author_name", ["Unknown Author"]))
    # Hint for subjects: book.get("subject", [])[:5]
    
    pass  # Remove this and implement the function

In [ ]:
# Test your fetch_book_data function
print("Testing fetch_book_data with 'The Great Gatsby'...")
try:
    book = fetch_book_data("The Great Gatsby")
    print(f"\nFound: {book['title']}")
    print(f"Author: {book['author']}")
    print(f"First Published: {book['year']}")
    print(f"Pages: {book['pages']}")
    print(f"Subjects: {', '.join(book['subjects'][:3]) if book['subjects'] else 'None'}")
    print("\nAPI helper working correctly!")
except Exception as e:
    print(f"Error: {e}")
    print("\nMake sure you've implemented all the TODOs in fetch_book_data()")

## Part 2: Create the Pydantic Validation Model

You are building a book information assistant. The agent should be able to look up details about books when users ask questions like "Tell me about 1984" or "Who wrote Pride and Prejudice?"

Implement a Pydantic model to validate book title queries. This protects your tool from malicious input and ensures data quality.

**Validation requirements for BookQuery:**
- Book title must be between 1 and 200 characters
- Title cannot be empty or just whitespace
- Title should only contain letters, numbers, spaces, hyphens, apostrophes, colons, and periods

In [ ]:
class BookQuery(BaseModel):
    """
    Validates a book title query.
    
    Attributes:
        title (str): The book title to search for (1-200 characters)
    
    Validation rules:
        - Title must be stripped of leading/trailing whitespace
        - Title cannot be empty after stripping
        - Only allow: letters, numbers, spaces, hyphens, apostrophes, colons, periods
    """
    title: str = Field(
        ...,
        min_length=1,
        max_length=200,
        description="The book title to search for"
    )
    
    @field_validator('title')
    @classmethod
    def validate_title(cls, v):
        """
        Sanitize and validate the book title.
        
        Should:
        1. Strip whitespace from the title
        2. Raise ValueError if title is empty after stripping
        3. Raise ValueError if title contains invalid characters
           (only allow: alphanumeric, space, hyphen, apostrophe, colon, period)
        4. Return the cleaned title
        """
        # TODO: Implement the validation logic
        # Hint: Use str.strip() to remove whitespace
        # Hint: Check each character with a loop or all() function
        # Hint: Valid characters: c.isalnum() or c in " -':."
        pass

## Part 3: Create the Book Information Tool

Implement a LangChain tool that fetches book information from the Open Library API with proper logging and error handling.

**Tool requirements:**
- Log when the tool is invoked with the book title
- Validate input using your Pydantic model
- Log success with execution duration
- Handle errors gracefully and return user-friendly messages
- Return a formatted string with book details (title, author, year, subjects)

In [ ]:
@tool
def get_book_info(title: str) -> str:
    """
    Get information about a book by its title using the Open Library API.
    
    Use this tool when the user asks about a specific book, wants to know
    who wrote a book, when it was published, or what it's about.
    
    Args:
        title (str): The title of the book to look up (e.g., "1984", "Pride and Prejudice")
        
    Returns:
        str: A formatted string containing:
             - Book title and publication year
             - Author name(s)
             - Number of pages (if available)
             - Subject categories
             
    Example output format:
        "Book: 1984 (1949)
         Author: George Orwell
         Pages: 328
         Subjects: Dystopian fiction, Political fiction, Science fiction"
    """
    # TODO 1: Log tool invocation with the title
    # Hint: logger.info(f"Tool invoked: get_book_info(title={title})")
    
    # TODO 2: Record start time for duration tracking
    # Hint: start_time = time.time()
    
    # TODO 3: Implement try/except block that:
    #   a) Validates input using BookQuery
    #   b) Fetches book data using fetch_book_data()
    #   c) Formats and returns the result string
    #   d) Logs success with duration
    #   e) Catches ValueError and returns user-friendly error message
    #   f) Catches TimeoutError and returns timeout message  
    #   g) Catches any other Exception and returns generic error message
    
    # Hint for formatting the result:
    # subjects_str = ", ".join(book_data['subjects']) if book_data['subjects'] else "Not available"
    # result = f"Book: {book_data['title']} ({book_data['year']})\n..."
    
    pass

## Test Your Validation Model

Run these tests to verify your Pydantic model works correctly. All tests should pass before moving on.

In [ ]:
# Test valid inputs
print("Testing valid inputs...")
try:
    query = BookQuery(title="1984")
    print(f"  Valid: '{query.title}'")
except Exception as e:
    print(f"  FAILED: {e}")

try:
    query = BookQuery(title="  The Great Gatsby  ")  # Should strip whitespace
    print(f"  Valid (stripped): '{query.title}'")
except Exception as e:
    print(f"  FAILED: {e}")

try:
    query = BookQuery(title="Harry Potter: The Philosopher's Stone")  # Apostrophe and colon
    print(f"  Valid (special chars): '{query.title}'")
except Exception as e:
    print(f"  FAILED: {e}")

# Test invalid inputs
print("\nTesting invalid inputs (should fail)...")

try:
    query = BookQuery(title="   ")  # Empty after strip
    print(f"  FAILED: Should have rejected whitespace-only title")
except ValueError as e:
    print(f"  Correctly rejected whitespace: OK")

try:
    query = BookQuery(title="Book<script>alert('xss')</script>")  # Invalid chars
    print(f"  FAILED: Should have rejected special characters")
except ValueError as e:
    print(f"  Correctly rejected special chars: OK")

try:
    query = BookQuery(title="Book; DROP TABLE books;--")  # SQL injection attempt
    print(f"  FAILED: Should have rejected SQL injection")
except ValueError as e:
    print(f"  Correctly rejected SQL injection: OK")

## Test Your Tool

Run these tests to verify your tool works correctly with the real Open Library API. Watch the logs to see the execution flow.

In [ ]:
# Test with a classic book (real API call!)
print("Testing get_book_info with '1984'...\n")
result = get_book_info.invoke({"title": "1984"})
print(f"Result:\n{result}")

In [ ]:
# Test with another book
print("Testing get_book_info with 'To Kill a Mockingbird'...\n")
result = get_book_info.invoke({"title": "To Kill a Mockingbird"})
print(f"Result:\n{result}")

In [ ]:
# Test with invalid input (should handle gracefully)
print("Testing get_book_info with invalid input...\n")
result = get_book_info.invoke({"title": "<script>bad</script>"})
print(f"Result:\n{result}")

## Create and Test the Agent

Create an agent with your book tool and test it with natural language queries.

In [ ]:
# Create the agent
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
book_agent = create_agent(model=model, tools=[get_book_info])

print("Book agent created successfully!")

In [ ]:
# Test the agent with a natural language query
query = "Tell me about the book 'The Hobbit'. Who wrote it?"
print(f"Query: {query}\n")

result = book_agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

print(f"Response: {result['messages'][-1].content}")

In [ ]:
# Test with another query
query = "What is 'Dune' about and when was it published?"
print(f"Query: {query}\n")

result = book_agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

print(f"Response: {result['messages'][-1].content}")